# 1. From Dataset to Image Encoder

In the previous notebook, we created our synthetic vision-language dataset.

Each batch of images is represented as a PyTorch tensor with shape:

$$
[B, 3, 32, 32]
$$

where:

- $B$ = batch size
- $3$ = RGB channels
- $32$ = image height
- $32$ = image width

The theoretical foundations notebook established that our image encoder should implement:

$$
z_I = f_I(I)
$$

where the image embedding is:

$$
z_I \in \mathbb{R}^{64}
$$

Therefore, the goal of this notebook is:

```text
[B, 3, 32, 32]
        ↓
  Image Encoder
        ↓
[B, 64]

# 1. Load the Dataset Pipeline

The image encoder should not recreate the synthetic dataset.

The dataset was already implemented in:

```text
01_dataset_exploration.ipynb

In [ ]:

# ============================================================
# Project setup
# ============================================================

from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"

# Make src importable
sys.path.insert(0, str(SRC_DIR))


# ------------------------------------------------------------
# Import dataset implementation
# ------------------------------------------------------------

from nano_vlm.data.dataset import SyntheticVLDataset


# ------------------------------------------------------------
# Artifact directories
# ------------------------------------------------------------

DATASET_DIR = PROJECT_ROOT / "artifacts" / "dataset"
FIGURES_DIR = PROJECT_ROOT / "assets" / "figures"
MODEL_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)
print("Device:", DEVICE)

Project root: d:\github\Build A NanoVLM From SCRATCH
Source directory: d:\github\Build A NanoVLM From SCRATCH\src
Device: cpu


In [29]:


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = PROJECT_ROOT / "artifacts" / "dataset"
FIGURES_DIR = PROJECT_ROOT / "assets" / "figures"
MODEL_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Load saved samples
# ------------------------------------------------------------

train_samples = torch.load(
    DATASET_DIR / "train_samples.pt",
    weights_only=False,
)

val_samples = torch.load(
    DATASET_DIR / "val_samples.pt",
    weights_only=False,
)

# ------------------------------------------------------------
# Recreate datasets
# ------------------------------------------------------------

train_dataset = SyntheticVLDataset(train_samples)
val_dataset = SyntheticVLDataset(val_samples)


# ------------------------------------------------------------
# Recreate DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("Device:", DEVICE)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Device: cpu
Train samples: 194
Validation samples: 49
Train batches: 7
Validation batches: 2


In [30]:
# Verify that the dataset DataLoader is now available

print(train_loader)


# 2. Why Use a CNN?

Images contain strong local spatial structure.

Nearby pixels are related to one another, and simple visual patterns can be combined into increasingly meaningful structures.

A CNN is useful because it can learn:

- **Edges** from local pixel patterns
- **Simple shapes** from combinations of edges
- **More complex visual features** from combinations of shapes
- **Spatial structure** through convolution and downsampling

CNNs also use **parameter sharing**.

Instead of learning a completely different set of parameters for every pixel location, the same convolutional filter is applied across the image.

For our tiny synthetic dataset, this is a very natural architecture.

We do not need a large vision transformer or a pretrained ResNet.

We want a small model whose operations are easy to inspect and understand.

The basic idea is:

```text
Pixels
  ↓
Local visual patterns
  ↓
Shapes and features
  ↓
Compact visual representation
  ↓
64-dimensional embedding



# 3. CNN Architecture

Our image encoder will use four convolutional layers.

The input is:

$$
32 \times 32 \times 3
$$

Each convolution uses:

- Kernel size: $3 \times 3$
- Stride: $2$
- Padding: $1$

The architecture is:

```text
Input
32 × 32 × 3

      ↓ Conv 3×3, stride 2

16 × 16 × 32

      ↓ Conv 3×3, stride 2

8 × 8 × 64

      ↓ Conv 3×3, stride 2

4 × 4 × 128

      ↓ Conv 3×3, stride 2

2 × 2 × 256

      ↓ Global Average Pooling

256

      ↓ Linear Projection

64-dimensional embedding

[B, 3, 32, 32]
        ↓
[B, 32, 16, 16]
        ↓
[B, 64, 8, 8]
        ↓
[B, 128, 4, 4]
        ↓
[B, 256, 2, 2]
        ↓
[B, 256, 1, 1]
        ↓
[B, 256]
        ↓
[B, 64]



# 4. Import PyTorch

We only need PyTorch for the image encoder.

We will implement the architecture ourselves using:

- `torch`
- `torch.nn`
- `torch.nn.functional`

No pretrained models are used.

In particular, we are not using:

- ResNet
- ViT
- CLIP
- Hugging Face vision models
- torchvision pretrained architectures

The purpose is to understand the image encoder from the individual layers upward.

# 5. Implement the Image Encoder

We can now implement our CNN.

The encoder contains four convolutional blocks.

Each block consists of:

```text
Conv2D
  ↓
ReLU

In [32]:

class ImageEncoder(nn.Module):
    """
    Small CNN-based image encoder for the NanoVLM.

    Input:
        [B, 3, 32, 32]

    Output:
        [B, 64]
    """

    def __init__(self, embedding_dim=64):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=2,
            padding=1,
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=2,
            padding=1,
        )

        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            stride=2,
            padding=1,
        )

        self.conv4 = nn.Conv2d(
            in_channels=128,
            out_channels=256,
            kernel_size=3,
            stride=2,
            padding=1,
        )

        # Global Average Pooling
        # [B, 256, 2, 2] -> [B, 256, 1, 1]
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Final projection into the shared embedding space
        self.projection = nn.Linear(
            256,
            embedding_dim,
        )

    def forward(self, x):
        # [B, 3, 32, 32]
        x = F.relu(self.conv1(x))

        # [B, 32, 16, 16]
        x = F.relu(self.conv2(x))

        # [B, 64, 8, 8]
        x = F.relu(self.conv3(x))

        # [B, 128, 4, 4]
        x = F.relu(self.conv4(x))

        # [B, 256, 2, 2]
        x = self.global_pool(x)

        # [B, 256, 1, 1] -> [B, 256]
        x = torch.flatten(x, start_dim=1)

        # [B, 256] -> [B, 64]
        x = self.projection(x)

        return x

# 6. Instantiate the Model

We can now create an instance of our image encoder.

The default embedding dimension is:

$$
64
$$

This must match the embedding dimension that our future text encoder will produce.

In [33]:
model = ImageEncoder(embedding_dim=64)

print(model)

ImageEncoder(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (global_pool): AdaptiveAvgPool2d(output_size=1)
  (projection): Linear(in_features=256, out_features=64, bias=True)
)


# 7. Why Global Average Pooling?

Before global average pooling, the final convolution produces:

$$
[B, 256, 2, 2]
$$

There are 256 feature channels, and each channel still has a small $2 \times 2$ spatial representation.

Global Average Pooling averages all spatial locations independently for each channel.

Therefore:

$$
[B, 256, 2, 2]
\rightarrow
[B, 256, 1, 1]
$$

After flattening:

$$
[B, 256, 1, 1]
\rightarrow
[B, 256]
$$

The important idea is that each of the 256 channels becomes one summary value.

So instead of keeping four spatial values per channel, we obtain one average value per channel.

This gives us a compact representation that can be passed directly into the final linear projection.

For our tiny model, this is simpler and more parameter-efficient than adding additional fully connected layers over the entire spatial feature map.

# 8. Why Project to 64 Dimensions?

After global average pooling, we have:

$$
[B, 256]
$$

But our shared multimodal embedding space has dimension:

$$
64
$$

Therefore we use:

$$
\text{Linear}(256 \rightarrow 64)
$$

which produces:

$$
[B, 64]
$$

The future text encoder will also produce:

$$
[B, 64]
$$

Therefore:

```text
Image Encoder                 Text Encoder

[B, 3, 32, 32]                Text Tokens
       ↓                           ↓
   CNN Encoder                Text Encoder
       ↓                           ↓
    [B, 64]                     [B, 64]
       \                           /
        \                         /
         └── Shared R^64 Space ──┘



# 9. Test the Encoder with a Dummy Batch

Before connecting the model to the dataset, we should test it using a synthetic PyTorch tensor.

Suppose the batch size is:

$$
B = 4
$$

Then the input should have shape:

$$
[4, 3, 32, 32]
$$

The encoder should produce:

$$
[4, 64]
$$

This simple test verifies that the architecture is connected correctly before we use real dataset images.

In [34]:
# Create a dummy batch

batch_size = 4

dummy_images = torch.randn(
    batch_size,
    3,
    32,
    32,
)

print("Input shape:", dummy_images.shape)

# Forward pass
dummy_embeddings = model(dummy_images)

print("Embedding shape:", dummy_embeddings.shape)

assert dummy_embeddings.shape == (4, 64)

print("✓ Output shape is correct.")

Input shape: torch.Size([4, 3, 32, 32])
Embedding shape: torch.Size([4, 64])
✓ Output shape is correct.


# 10. Inspect Intermediate Shapes

Understanding tensor shapes is one of the most important skills when building CNNs.

For our encoder, the expected progression is:

```text
Input
[B, 3, 32, 32]

        ↓

Conv 1
[B, 32, 16, 16]

        ↓

Conv 2
[B, 64, 8, 8]

        ↓

Conv 3
[B, 128, 4, 4]

        ↓

Conv 4
[B, 256, 2, 2]

        ↓

Global Average Pooling
[B, 256, 1, 1]

        ↓

Flatten
[B, 256]

        ↓

Linear Projection
[B, 64]

In [35]:


# Inspect intermediate tensor shapes

x = dummy_images

print("Input:                  ", x.shape)

x = F.relu(model.conv1(x))
print("After Conv 1:           ", x.shape)

x = F.relu(model.conv2(x))
print("After Conv 2:           ", x.shape)

x = F.relu(model.conv3(x))
print("After Conv 3:           ", x.shape)

x = F.relu(model.conv4(x))
print("After Conv 4:           ", x.shape)

x = model.global_pool(x)
print("After Global Pooling:   ", x.shape)

x = torch.flatten(x, start_dim=1)
print("After Flatten:          ", x.shape)

x = model.projection(x)
print("After Projection:       ", x.shape)

Input:                   torch.Size([4, 3, 32, 32])
After Conv 1:            torch.Size([4, 32, 16, 16])
After Conv 2:            torch.Size([4, 64, 8, 8])
After Conv 3:            torch.Size([4, 128, 4, 4])
After Conv 4:            torch.Size([4, 256, 2, 2])
After Global Pooling:    torch.Size([4, 256, 1, 1])
After Flatten:           torch.Size([4, 256])
After Projection:        torch.Size([4, 64])


## Shape Verification

The observed shapes should be:

$$
[B, 3, 32, 32]
$$

$$
\downarrow
$$

$$
[B, 32, 16, 16]
$$

$$
\downarrow
$$

$$
[B, 64, 8, 8]
$$

$$
\downarrow
$$

$$
[B, 128, 4, 4]
$$

$$
\downarrow
$$

$$
[B, 256, 2, 2]
$$

$$
\downarrow
$$

$$
[B, 256, 1, 1]
$$

$$
\downarrow
$$

$$
[B, 256]
$$

$$
\downarrow
$$

$$
[B, 64]
$$

This confirms that every major stage produces the expected tensor dimensions.

# 11. Parameter Count

The number of trainable parameters tells us how large our model actually is.

We can calculate it with:

In [36]:
sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

404864

In [37]:

# Count trainable parameters

num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {num_parameters:,}")

Trainable parameters: 404,864


# 12. Inspect the Parameters by Layer

The total parameter count is useful, but looking at the individual layers gives us more intuition.

Each convolution contains:

- Weights
- Biases

The final projection layer also contains weights and biases.

Let's inspect the parameter count for each trainable tensor.

In [38]:
# Parameter breakdown

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(
            f"{name:30s} "
            f"shape={str(tuple(parameter.shape)):20s} "
            f"parameters={parameter.numel():,}"
        )

conv1.weight                   shape=(32, 3, 3, 3)        parameters=864
conv1.bias                     shape=(32,)                parameters=32
conv2.weight                   shape=(64, 32, 3, 3)       parameters=18,432
conv2.bias                     shape=(64,)                parameters=64
conv3.weight                   shape=(128, 64, 3, 3)      parameters=73,728
conv3.bias                     shape=(128,)               parameters=128
conv4.weight                   shape=(256, 128, 3, 3)     parameters=294,912
conv4.bias                     shape=(256,)               parameters=256
projection.weight              shape=(64, 256)            parameters=16,384
projection.bias                shape=(64,)                parameters=64


# 13. Test on a Real Batch from the Dataset

The previous dataset notebook provides a `train_loader`.

We do not recreate the dataset here.

Instead, we use the existing DataLoader interface.

A training batch should provide images with shape:

$$
[B, 3, 32, 32]
$$

We pass those images into our encoder:

```text
Images
[B, 3, 32, 32]
      ↓
Image Encoder
      ↓
Embeddings
[B, 64]

In [39]:

# Get one real batch from the dataset DataLoader
# Get one real batch from the dataset

batch = next(iter(train_loader))

images = batch["image"]
captions = batch["caption"]

print("Images shape:", images.shape)
print("Number of captions:", len(captions))


Images shape: torch.Size([32, 3, 32, 32])
Number of captions: 32


> **If the dataset notebook returns a dictionary instead of `(images, captions)`, use the corresponding keys from that notebook.**

For example:

In [40]:

batch = next(iter(train_loader))

images = batch["image"]
captions = batch["caption"]

In [41]:


# Generate image embeddings from the real dataset batch

model.eval()

with torch.no_grad():
    image_embeddings = model(images)

print("Images shape:           ", images.shape)
print("Image embeddings shape:", image_embeddings.shape)

assert images.ndim == 4
assert images.shape[1:] == (3, 32, 32)

assert image_embeddings.ndim == 2
assert image_embeddings.shape[0] == images.shape[0]
assert image_embeddings.shape[1] == 64

print("✓ Real batch successfully passed through the image encoder.")

Images shape:            torch.Size([32, 3, 32, 32])
Image embeddings shape: torch.Size([32, 64])
✓ Real batch successfully passed through the image encoder.


# 14. Inspect Raw Embeddings

Let's look at a few image embeddings.

Each image is now represented by a vector with 64 values:

$$
z_I \in \mathbb{R}^{64}
$$

For a batch, this becomes:

$$
Z_I \in \mathbb{R}^{B \times 64}
$$

However, there is an important point:

> **These embeddings are currently untrained.**

The encoder has only been initialized with random parameters.

Therefore, we should **not** expect the embedding vectors to have meaningful semantic structure yet.

For example, we should not expect all red circles to be close together.

Meaningful visual representations will emerge later when the encoder is trained jointly with the text encoder using the contrastive objective.

In [42]:
# Inspect the first two raw image embeddings

print(image_embeddings[:2])

tensor([[-0.0153, -0.0024,  0.0521,  0.0477,  0.0300, -0.0662,  0.0373, -0.0604,
          0.0203,  0.0334,  0.0250, -0.0236,  0.0149,  0.0376, -0.0148, -0.0473,
         -0.0400, -0.0326,  0.0063, -0.0403, -0.0336,  0.0133, -0.0102,  0.0163,
         -0.0531, -0.0449,  0.0150, -0.0039, -0.0700,  0.0399, -0.0428,  0.0361,
         -0.0272,  0.0450, -0.0332, -0.0018,  0.0396,  0.0450,  0.0040, -0.0454,
         -0.0447,  0.0020, -0.0021,  0.0394, -0.0586, -0.0423,  0.0021, -0.0349,
          0.0485, -0.0221,  0.0046,  0.0176,  0.0270, -0.0150,  0.0453, -0.0194,
         -0.0371, -0.0417, -0.0061, -0.0024,  0.0546,  0.0022, -0.0378, -0.0207],
        [-0.0156, -0.0024,  0.0528,  0.0474,  0.0294, -0.0661,  0.0361, -0.0615,
          0.0211,  0.0332,  0.0240, -0.0240,  0.0150,  0.0377, -0.0150, -0.0469,
         -0.0389, -0.0320,  0.0054, -0.0409, -0.0336,  0.0132, -0.0109,  0.0160,
         -0.0533, -0.0459,  0.0143, -0.0047, -0.0704,  0.0400, -0.0421,  0.0366,
         -0.0273,  0.0443, 

# 15. Embedding Normalization

The contrastive-learning stage will eventually compare image and text embeddings.

Before calculating cosine similarity, we will normalize the embeddings.

For an embedding $z$:

$$
\hat{z}
=
\frac{z}{\|z\|_2}
$$

This makes:

$$
\|\hat{z}\|_2 = 1
$$

In PyTorch, this can be done with:

```python
F.normalize(z, dim=-1)

In [43]:

# Demonstrate L2 normalization

normalized_embeddings = F.normalize(
    image_embeddings,
    dim=-1,
)

print("Normalized embedding shape:", normalized_embeddings.shape)

# Calculate the L2 norm of each embedding
embedding_norms = torch.norm(
    normalized_embeddings,
    dim=-1,
)

print("\nEmbedding norms:")
print(embedding_norms)

# The norms should be approximately 1
assert torch.allclose(
    embedding_norms,
    torch.ones_like(embedding_norms),
    atol=1e-6,
)

print("\n✓ Embeddings are L2-normalized.")

Normalized embedding shape: torch.Size([32, 64])

Embedding norms:
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

✓ Embeddings are L2-normalized.


# 16. Connection to the Contrastive Learning Pipeline

Our image encoder is now one component of the larger NanoVLM.

The complete image-side pipeline is:

```text
Image
  ↓
Image Encoder
  ↓
64-dimensional Image Embedding
  ↓
L2 Normalization
  ↓
Similarity with Text Embedding
  ↓
Similarity Matrix
  ↓
Contrastive Loss


# 17. Why We Are Not Visualizing the Embeddings Yet

It may be tempting to immediately apply PCA, t-SNE, or UMAP to the embeddings.

However, the current embeddings are **untrained**.

The CNN parameters have not yet learned the relationship between:

- colors
- shapes
- positions
- captions

Therefore, a visualization of these embeddings would not tell us much about semantic learning.

Later, after contrastive training, dimensionality reduction can become useful for investigating whether visually and linguistically related concepts occupy nearby regions of the shared embedding space.

For now, our priority is verifying that the encoder architecture works correctly.

# 18. Final Architecture

The complete image encoder can be summarized as:

```text
Input Image
[B, 3, 32, 32]
       │
       ▼
Conv2D(3 → 32)
       │
       ▼
[B, 32, 16, 16]
       │
       ▼
Conv2D(32 → 64)
       │
       ▼
[B, 64, 8, 8]
       │
       ▼
Conv2D(64 → 128)
       │
       ▼
[B, 128, 4, 4]
       │
       ▼
Conv2D(128 → 256)
       │
       ▼
[B, 256, 2, 2]
       │
       ▼
Global Average Pooling
       │
       ▼
[B, 256, 1, 1]
       │
       ▼
Flatten
       │
       ▼
[B, 256]
       │
       ▼
Linear(256 → 64)
       │
       ▼
[B, 64]

In [44]:
# ============================================================
# Save Image Encoder artifacts and outputs
# ============================================================

MODEL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Save the initial Image Encoder state
# ------------------------------------------------------------

image_encoder_path = (
    MODEL_ARTIFACTS_DIR / "image_encoder_initial.pt"
)

torch.save(
    model.state_dict(),
    image_encoder_path,
)


# ------------------------------------------------------------
# Save a representative embedding batch
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    # DataLoader returns a dictionary batch
    batch = next(iter(train_loader))

    # Extract only the images needed by the Image Encoder
    images = batch["image"].to(DEVICE)

    image_embeddings = model(images)

# ------------------------------------------------------------
# Verify expected output
# ------------------------------------------------------------

print("Images shape:", images.shape)
print("Image embeddings shape:", image_embeddings.shape)

assert images.ndim == 4
assert images.shape[1:] == (3, 32, 32)

assert image_embeddings.ndim == 2
assert image_embeddings.shape[1] == 64


# ------------------------------------------------------------
# Save representative embeddings
# ------------------------------------------------------------

torch.save(
    image_embeddings.cpu(),
    MODEL_ARTIFACTS_DIR / "image_embeddings_initial.pt",
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("\nImage Encoder artifacts saved:")

print("✓", image_encoder_path)
print(
    "✓",
    MODEL_ARTIFACTS_DIR / "image_embeddings_initial.pt"
)

Images shape: torch.Size([32, 3, 32, 32])
Image embeddings shape: torch.Size([32, 64])

Image Encoder artifacts saved:
✓ d:\github\Build A NanoVLM From SCRATCH\artifacts\models\image_encoder_initial.pt
✓ d:\github\Build A NanoVLM From SCRATCH\artifacts\models\image_embeddings_initial.pt




# 19. Final Summary

We have now implemented the image encoder for our NanoVLM completely from scratch.

The encoder contains:

- Four convolutional layers
- ReLU activations
- Spatial downsampling
- Increasing feature channels
- Global Average Pooling
- A linear projection layer
- A 64-dimensional output embedding

The complete transformation is:

$$
32 \times 32 \times 3
\rightarrow
\text{CNN}
\rightarrow
256\text{-dimensional representation}
\rightarrow
64\text{-dimensional embedding}
$$

In batch form:

$$
[B,3,32,32]
\rightarrow
[B,64]
$$

The resulting vector represents the image in the shared embedding space:

$$
z_I \in \mathbb{R}^{64}
$$

At this stage, the embeddings are still **untrained**.

The next major component is the **text encoder**.

It will eventually transform captions into:

$$
z_T \in \mathbb{R}^{64}
$$

After both encoders are implemented, we will be able to connect them using the contrastive-learning objective introduced in:

```text
00_theoretical_foundations.ipynb

In [45]:

# Final sanity check

model.eval()

with torch.no_grad():
    final_embeddings = model(images)

print("=" * 60)
print("NanoVLM Image Encoder - Final Sanity Check")
print("=" * 60)

print(f"Input shape:          {tuple(images.shape)}")
print(f"Output shape:         {tuple(final_embeddings.shape)}")
print(f"Embedding dimension:  {final_embeddings.shape[-1]}")
print(f"Trainable parameters: {num_parameters:,}")

assert images.ndim == 4
assert images.shape[1:] == (3, 32, 32)

assert final_embeddings.ndim == 2
assert final_embeddings.shape[0] == images.shape[0]
assert final_embeddings.shape[1] == 64

print("\n✓ Input shape verified.")
print("✓ Output shape verified.")
print("✓ Embedding dimension verified.")
print("✓ Image encoder is ready for the text-encoder stage.")

NanoVLM Image Encoder - Final Sanity Check
Input shape:          (32, 3, 32, 32)
Output shape:         (32, 64)
Embedding dimension:  64
Trainable parameters: 404,864

✓ Input shape verified.
✓ Output shape verified.
✓ Embedding dimension verified.
✓ Image encoder is ready for the text-encoder stage.
